# Regularization

In the Polynomial Regression lab we saw that a model that is too flexible **overfits**: it memorises the noise of the training data and fails on new data. We controlled the problem by choosing a smaller degree. But what if we *want* to use a flexible model (many features) and still avoid overfitting?

**Regularization** is the answer: we keep the flexible model but add a **penalty on the size of the weights** to the cost function. Large weights are what makes a curve wiggle wildly, so penalising them forces the model to be smooth.

**In this lab you will:**
1. See overfitting again, and how it shows up in the size of the weights.
2. Add an L2 penalty (*ridge regression*) to the cost function and to gradient descent.
3. Study the effect of the regularization strength $\lambda$ and choose it with a validation set and with cross-validation.
4. Apply exactly the same idea to **logistic regression**, a classifier.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (7, 4.5)

## 1. Overfitting: the problem

We will use a classic example: 20 noisy samples of $y = \sin(2\pi x)$ for $x \in [0, 1]$. This is not a polynomial, so no polynomial fits it exactly, and a high degree will be tempted to chase the noise. We also generate 200 extra points that we will use as a **validation set**.

In [ ]:
def make_sine(n, seed, noise=0.25):
    r = np.random.default_rng(seed)
    x = np.sort(r.uniform(0, 1, n))
    return x, np.sin(2 * np.pi * x) + r.normal(0, noise, n)


x_train, y_train = make_sine(20, seed=2)
x_val, y_val = make_sine(200, seed=102)

xs = np.linspace(0, 1, 300)
plt.scatter(x_train, y_train, edgecolor="k", label="Training data")
plt.plot(xs, np.sin(2 * np.pi * xs), "g--", label="True function")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

We use polynomial features again. Here $x$ is simply rescaled to $[-1, 1]$ (so that all the powers have a similar size), and we use degree 9: a very flexible model for only 20 points.

In [ ]:
def map_feature(x, degree):
    z = 2 * x - 1                                            # rescale x from [0, 1] to [-1, 1]
    return np.column_stack([z ** k for k in range(degree + 1)])


def rmse(y, y_pred):
    return np.sqrt(np.mean((y - y_pred) ** 2))


degree = 9
X_train, X_val = map_feature(x_train, degree), map_feature(x_val, degree)

w_ols, *_ = np.linalg.lstsq(X_train, y_train, rcond=None)      # plain least squares, no regularization

print("Training RMSE:  ", round(rmse(y_train, X_train @ w_ols), 3))
print("Validation RMSE:", round(rmse(y_val, X_val @ w_ols), 3))
print("Sum of squared weights (without the bias):", round(np.sum(w_ols[1:] ** 2)))

plt.scatter(x_train, y_train, edgecolor="k", label="Training data")
plt.plot(xs, np.sin(2 * np.pi * xs), "g--", label="True function")
plt.plot(xs, map_feature(xs, degree) @ w_ols, "r", label="Degree 9, no regularization")
plt.ylim(-2, 2)
plt.legend()
plt.show()

The model overfits: tiny training error, large validation error, and a curve that oscillates wildly between the points. Notice the **sum of squared weights**: it is enormous. To make such a wiggly curve the model needs huge positive and negative weights that nearly cancel each other out. This suggests how to fight overfitting: *do not let the weights grow*.

## 2. The idea: penalise large weights

We add a penalty term to the cost function. With $m$ training examples and $d$ features (plus the bias $w_0$):

$$J(\mathbf{w}) = \frac{1}{2m}\left[\sum_{i=1}^{m}\left(\hat{y}_i - y_i\right)^2 + \lambda \sum_{j=1}^{d} w_j^2\right]$$

* The first term is the usual squared error: *fit the data*.
* The second term is the **L2 penalty**: *keep the weights small*.
* $\lambda \geq 0$ is the **regularization strength**, a *hyperparameter* we choose. $\lambda = 0$ is the ordinary model; the larger $\lambda$ is, the smoother (simpler) the model becomes.
* The bias $w_0$ is **not** penalised (the sum starts at $j = 1$): shifting the whole curve up or down does not make it more complex.

The gradient only gets one extra term (the derivative of $\frac{\lambda}{2m}w_j^2$):

$$\frac{\partial J}{\partial w_j} = \frac{1}{m}\left[\sum_{i=1}^{m}\left(\hat{y}_i - y_i\right)x_{ij} + \lambda\, w_j\right] \quad (j \geq 1), \qquad \frac{\partial J}{\partial w_0} = \frac{1}{m}\sum_{i=1}^{m}\left(\hat{y}_i - y_i\right)x_{i0}$$

> **Note on conventions.** In the *Problems 6* notebook the penalty was written as $\lambda \sum w_j^2$ inside a cost divided by $m$, so its gradient had $2\lambda w_j$. That $\lambda$ is **half** of the one used here: $\lambda = 0.5$ there corresponds to $\lambda = 1$ in this lab. The shape of the results is the same.

## 3. Ridge regression: two ways to solve it

Least squares with an L2 penalty is called **ridge regression**. It can be solved in two ways.

**a) Closed form.** Setting the gradient to zero gives the *regularized normal equations*:

$$\left(X^T X + \lambda P\right)\mathbf{w} = X^T \mathbf{y}$$

where $P$ is the identity matrix with a $0$ in the first position (so the bias is not penalised).

In [ ]:
def ridge_closed_form(X, y, lam):
    penalty = np.eye(X.shape[1])
    penalty[0, 0] = 0                       # do not penalise the bias
    return np.linalg.solve(X.T @ X + lam * penalty, X.T @ y)

**b) Gradient descent.** The same algorithm as always, with the extra term in the gradient. We also record the cost so that we can check that it goes down.

In [ ]:
def gradient_descent_ridge(X, y, lam, alpha=0.5, iterations=20000):
    m = len(y)
    w = np.zeros(X.shape[1])
    cost = []
    for _ in range(iterations):
        w_pen = w.copy()
        w_pen[0] = 0                         # the bias is not penalised
        error = X @ w - y
        cost.append((error @ error + lam * (w_pen @ w_pen)) / (2 * m))
        w = w - alpha * (X.T @ error + lam * w_pen) / m
    return w, np.array(cost)

In [ ]:
lam = 0.1
w_closed = ridge_closed_form(X_train, y_train, lam)
w_gd, cost = gradient_descent_ridge(X_train, y_train, lam)

print("Largest difference between both solutions:", np.abs(w_closed - w_gd).max())

plt.semilogy(cost)
plt.xlabel("Iteration")
plt.ylabel("Regularized cost")
plt.show()

Both methods give the same weights. For this small problem the closed form is faster and exact, but gradient descent is the one that generalises to models without a closed-form solution, such as **logistic regression** (section 6).

## 4. The effect of $\lambda$

Let's fit the degree-9 model with different values of $\lambda$. Note how the curve becomes smoother and the weights smaller as $\lambda$ grows, until the model is so constrained that it can no longer follow the data.

In [ ]:
lambdas_demo = [0, 1e-4, 1e-2, 1, 100]

fig, axes = plt.subplots(1, len(lambdas_demo), sharey=True, figsize=(20, 4))
for lam, ax in zip(lambdas_demo, axes):
    w = ridge_closed_form(X_train, y_train, lam)
    ax.scatter(x_train, y_train, edgecolor="k")
    ax.plot(xs, np.sin(2 * np.pi * xs), "g--")
    ax.plot(xs, map_feature(xs, degree) @ w, "r")
    ax.set_ylim(-2, 2)
    ax.set_xlabel("x")
    ax.set_title(f"lambda = {lam}\n|w|^2 = {np.sum(w[1:] ** 2):.1f} | val RMSE {rmse(y_val, X_val @ w):.2f}")
plt.show()

In [ ]:
print(f"{'lambda':>8} | {'sum w^2':>12} | {'train RMSE':>10} | {'val RMSE':>8}")
for lam in [0, 1e-6, 1e-4, 1e-2, 1e-1, 1, 10, 100]:
    w = ridge_closed_form(X_train, y_train, lam)
    print(f"{lam:>8} | {np.sum(w[1:] ** 2):12.2f} | {rmse(y_train, X_train @ w):10.3f} | {rmse(y_val, X_val @ w):8.3f}")

* With $\lambda = 0$ the weights are huge and the validation error is large: **overfitting**.
* With a small-to-moderate $\lambda$ the weights shrink by orders of magnitude, the curve follows the true function, and the validation error drops.
* With a very large $\lambda$ the weights are pushed to (almost) zero and the model becomes a flat line: **underfitting**. Both training and validation errors go up again.

$\lambda$ plays the role that the degree played in the previous lab, but it is a **continuous** knob instead of a discrete one.

## 5. Choosing $\lambda$

### 5.1 With a validation set

We try many values of $\lambda$ (on a logarithmic grid) and keep the one with the lowest validation error. The training error is useless here: it always prefers $\lambda = 0$.

In [ ]:
lambdas = np.logspace(-6, 2, 30)
train_err = [rmse(y_train, X_train @ ridge_closed_form(X_train, y_train, lam)) for lam in lambdas]
val_err = [rmse(y_val, X_val @ ridge_closed_form(X_train, y_train, lam)) for lam in lambdas]

best_val = lambdas[int(np.argmin(val_err))]
print("Best lambda according to the validation set:", round(best_val, 5))

plt.semilogx(lambdas, train_err, "o-", label="Training")
plt.semilogx(lambdas, val_err, "o-", label="Validation")
plt.axvline(best_val, color="gray", linestyle="--")
plt.xlabel("lambda (log scale)")
plt.ylabel("RMSE")
plt.legend()
plt.show()

### 5.2 With cross-validation

A validation set of 200 points is a luxury: in real problems we often only have the 20 training points. **k-fold cross-validation** solves this: we split the training data into $k$ folds, and for each fold we train on the other $k-1$ folds and measure the error on the one left out. The average of the $k$ errors is our estimate.

In [ ]:
def kfold_error(x, y, degree, lam, k=5, seed=0):
    idx = np.random.default_rng(seed).permutation(len(x))
    folds = np.array_split(idx, k)
    errors = []
    for i in range(k):
        val_idx = folds[i]
        train_idx = np.concatenate([folds[j] for j in range(k) if j != i])
        w = ridge_closed_form(map_feature(x[train_idx], degree), y[train_idx], lam)
        errors.append(rmse(y[val_idx], map_feature(x[val_idx], degree) @ w))
    return np.mean(errors)


cv_err = [kfold_error(x_train, y_train, degree, lam) for lam in lambdas]
best_cv = lambdas[int(np.argmin(cv_err))]
print("Best lambda according to 5-fold cross-validation:", round(best_cv, 5))

w_cv = ridge_closed_form(X_train, y_train, best_cv)
print("Validation RMSE of the model chosen by cross-validation:", round(rmse(y_val, X_val @ w_cv), 3))

plt.semilogx(lambdas, cv_err, "o-", label="5-fold cross-validation (20 points)")
plt.semilogx(lambdas, val_err, "o-", label="Large validation set (200 points)")
plt.axvline(best_cv, color="gray", linestyle="--")
plt.xlabel("lambda (log scale)")
plt.ylabel("RMSE")
plt.legend()
plt.show()

Using **only the 20 training points**, cross-validation selects a $\lambda$ in the same region as the large validation set, and the resulting model is good. This is how hyperparameters are chosen in practice.

## 6. Regularized logistic regression

The same idea works for classifiers. Recall logistic regression: $\hat{y} = \sigma(\mathbf{w}^T\mathbf{x})$ where $\sigma(z) = \frac{1}{1+e^{-z}}$, trained by minimising the **log loss**. With an L2 penalty:

$$J(\mathbf{w}) = -\frac{1}{m}\sum_{i=1}^{m}\left[y_i\log\hat{y}_i + (1-y_i)\log(1-\hat{y}_i)\right] + \frac{\lambda}{2m}\sum_{j=1}^{d} w_j^2$$

and the gradient is **the same extra term as before**:

$$\frac{\partial J}{\partial w_j} = \frac{1}{m}\left[\sum_{i=1}^{m}\left(\hat{y}_i - y_i\right)x_{ij} + \lambda\, w_j\right] \quad (j \geq 1)$$

(again, no penalty for $w_0$). Let's see it in action on a dataset that is **not linearly separable**: two interleaving half-moons with noise. To draw a curved decision boundary with a linear model we need polynomial features of the two variables, which makes the model very flexible and prone to overfit when there are few points.

In [ ]:
def make_moons(n, noise, rng):
    n1 = n // 2
    n2 = n - n1
    t1 = rng.uniform(0, np.pi, n1)
    t2 = rng.uniform(0, np.pi, n2)
    upper = np.column_stack([np.cos(t1), np.sin(t1)])
    lower = np.column_stack([1 - np.cos(t2), 0.5 - np.sin(t2)])
    X = np.vstack([upper, lower]) + rng.normal(0, noise, (n, 2))
    y = np.r_[np.zeros(n1), np.ones(n2)]
    p = rng.permutation(n)
    return X[p], y[p]


rng = np.random.default_rng(3)
X2_train, y2_train = make_moons(40, 0.35, rng)       # few training points
X2_val, y2_val = make_moons(1000, 0.35, rng)         # large validation set

plt.scatter(X2_train[y2_train == 0, 0], X2_train[y2_train == 0, 1], marker="o", c="b", edgecolor="k", label="Class 0")
plt.scatter(X2_train[y2_train == 1, 0], X2_train[y2_train == 1, 1], marker="x", c="r", label="Class 1")
plt.xlabel("x1")
plt.ylabel("x2")
plt.legend()
plt.show()

The function below builds all the terms $x_1^{a} x_2^{b}$ with $1 \le a + b \le$ `degree` (for degree 10 that is 65 features, plus the bias) and standardises each column using the training statistics.

In [ ]:
def poly2d(X, degree, mu=None, sigma=None):
    F = np.column_stack([X[:, 0] ** (i - j) * X[:, 1] ** j for i in range(1, degree + 1) for j in range(i + 1)])
    if mu is None:
        mu, sigma = F.mean(axis=0), F.std(axis=0)
    return np.column_stack([np.ones(len(X)), (F - mu) / sigma]), mu, sigma


def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))


def logistic_cost(X, y, w, lam):
    p = sigmoid(X @ w)
    w_pen = w.copy()
    w_pen[0] = 0
    eps = 1e-12
    return -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps)) + lam / (2 * len(y)) * (w_pen @ w_pen)


def gradient_descent_logistic(X, y, lam, alpha=0.5, iterations=8000):
    m = len(y)
    w = np.zeros(X.shape[1])
    cost = []
    for _ in range(iterations):
        w_pen = w.copy()
        w_pen[0] = 0
        cost.append(logistic_cost(X, y, w, lam))
        w = w - alpha * (X.T @ (sigmoid(X @ w) - y) + lam * w_pen) / m
    return w, np.array(cost)


def accuracy(X, y, w):
    return np.mean((sigmoid(X @ w) > 0.5) == y)

We train a degree-10 model for several values of $\lambda$ and compare the accuracy on the training set and on the (large) validation set.

In [ ]:
degree2 = 10
F_train, mu2, sigma2 = poly2d(X2_train, degree2)
F_val, _, _ = poly2d(X2_val, degree2, mu2, sigma2)
print("Number of features (including the bias):", F_train.shape[1], "| training points:", len(y2_train))

lams = [0, 0.1, 1, 10, 30]
models = {}
print(f"{'lambda':>6} | {'train acc':>9} | {'val acc':>7} | {'sum w^2':>9}")
for lam in lams:
    w, _ = gradient_descent_logistic(F_train, y2_train, lam)
    models[lam] = w
    print(f"{lam:>6} | {accuracy(F_train, y2_train, w):9.3f} | {accuracy(F_val, y2_val, w):7.3f} | {np.sum(w[1:] ** 2):9.1f}")

Without regularization ($\lambda = 0$) the model gets (almost) all the training points right but is clearly worse on new data. A moderate $\lambda$ gives up some training accuracy and **gains validation accuracy**; a too-large $\lambda$ underfits.

Let's look at the decision boundaries (the red/blue regions show what the model predicts at each point of the plane):

In [ ]:
gx, gy = np.meshgrid(np.linspace(-1.8, 2.8, 250), np.linspace(-1.5, 2.0, 250))
grid = np.column_stack([gx.ravel(), gy.ravel()])
F_grid, _, _ = poly2d(grid, degree2, mu2, sigma2)

fig, axes = plt.subplots(1, len(lams), sharey=True, figsize=(20, 4))
for lam, ax in zip(lams, axes):
    w = models[lam]
    prob = sigmoid(F_grid @ w).reshape(gx.shape)
    ax.contourf(gx, gy, prob, levels=[0, 0.5, 1], colors=["#cfe0ff", "#ffd6d6"])
    ax.contour(gx, gy, prob, levels=[0.5], colors="k")
    ax.scatter(X2_train[y2_train == 0, 0], X2_train[y2_train == 0, 1], marker="o", c="b", edgecolor="k")
    ax.scatter(X2_train[y2_train == 1, 0], X2_train[y2_train == 1, 1], marker="x", c="r")
    ax.set_title(f"lambda = {lam}\ntrain {accuracy(F_train, y2_train, w):.2f} | val {accuracy(F_val, y2_val, w):.2f}")
    ax.set_xlabel("x1")
axes[0].set_ylabel("x2")
plt.show()

With $\lambda = 0$ the boundary twists around individual points (even carving small islands to catch isolated points). As $\lambda$ grows the boundary becomes smooth, and for a very large $\lambda$ it degenerates into something too simple.

## 7. Notes

* **Always standardise the features before regularizing.** The penalty $\sum w_j^2$ treats all weights alike; if one feature is measured in much larger units its weight will be naturally tiny and barely penalised, while the weights of small-scale features would be over-penalised. That is why every function above standardises its features.
* **Other penalties.** The **L1 penalty** (*lasso*), $\lambda \sum |w_j|$, tends to set some weights exactly to zero, performing feature selection. It has no closed form but can be trained with variations of gradient descent.
* **In scikit-learn:** `Ridge(alpha=...)` and `Lasso(alpha=...)` for regression (their `alpha` plays the role of our $\lambda$), and `LogisticRegression(C=...)`, where $C = 1/\lambda$. Beware: `LogisticRegression` is regularized **by default** (`C=1.0`).

## Exercises

### Exercise 1: a simpler model

Repeat the analysis of section 5.1 but with a **degree-5** polynomial instead of degree 9: compute the validation RMSE for the values in `lambdas` and compare the best result with the one obtained with $\lambda = 0$. Does regularization help as much as it did with degree 9? Why do you think that is?

*Hint: build `X_train5 = map_feature(x_train, 5)` and `X_val5 = map_feature(x_val, 5)` and reuse `ridge_closed_form` and `rmse`.*

In [ ]:
# Your Code Here

---

*Your Answer Here*

---

### Exercise 2: a different degree for the classifier

Repeat the experiment of section 6 with a **degree-6** polynomial (`degree2 = 6`). Build the feature matrices, train the model for $\lambda \in \{0, 0.1, 1, 10, 30\}$ and print a table with training accuracy, validation accuracy and $\sum w_j^2$. Which $\lambda$ would you choose? Is the effect of regularization as dramatic as with degree 10?

*Hint: reuse `poly2d`, `gradient_descent_logistic` and `accuracy`.*

In [ ]:
# Your Code Here

---

*Your Answer Here*

---

### Exercise 3: questions

(a) Why is the bias $w_0$ not penalised?

(b) What happens to the model when $\lambda \to 0$? And when $\lambda \to \infty$?

(c) Why would it be a mistake to choose $\lambda$ using the **training** error?

(d) You train a regularized model on features where one of them is the *price in euros* (values of order 100000) and the others are between 0 and 1, without standardising. What problem would you expect?

---

*Your Answer Here*

---